# W2D1 — The Data Toolkit — Guided

**Week 2 · Day 1 · Data Engineering & Preprocessing** · Lab

Last week the data arrived ready to model. It never does. Today you take a raw customer table and
turn it into a **feature table**: one row per customer, one column per input, no target hiding in the
features.

You will use six pandas operations, and they cover most of the data work you will ever do: selection,
boolean filtering, `groupby` + `agg`, `merge`, assigning new columns, and doing arithmetic on whole
columns at once instead of row by row.

The best moment of the lab is small and early. One column in this file is text when it should be a
number, and converting it produces eleven missing values. Those eleven are not noise — they have a
cause, and finding it is the difference between cleaning data and guessing at it.

You'll leave with `features.parquet`, which D2, D3, D4 and D5 all load.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ١ — أدوات البيانات

**الأسبوع ٢ · اليوم ١ · هندسة البيانات والمعالجة المسبقة** · معمل عملي

في الأسبوع الماضي وصلتك البيانات جاهزة للنمذجة، وهذا لا يحدث في الواقع. اليوم تأخذ جدول عملاء خامًا
وتحوّله إلى **جدول خصائص**: صف واحد لكل عميل، وعمود واحد لكل مُدخل، ولا هدف مخبّأ بين الخصائص.

ستستخدم ستّ عمليات في pandas تغطّي معظم عمل البيانات الذي ستقوم به: الاختيار، والترشيح المنطقي،
و`groupby` مع `agg`، و`merge`، وإضافة أعمدة جديدة، وإجراء الحساب على العمود كاملًا مرة واحدة بدل
المرور على الصفوف صفًا صفًا.

وأجمل لحظة في المعمل صغيرة وتأتي مبكرًا: عمود في هذا الملف مخزّن كنص وكان يجب أن يكون رقمًا، وتحويله
يُنتج إحدى عشرة قيمة مفقودة. وهذه القيم ليست ضوضاء، بل لها سبب، والعثور على السبب هو الفرق بين تنظيف
البيانات والتخمين فيها.

ستخرج بملف `features.parquet` الذي تُحمّله أيام الأسبوع الثاني والثالث والرابع والخامس.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Read a table's `dtypes` and spot a column whose type is wrong before it breaks a model.
- Convert a text column to numeric, and explain what the resulting missing values mean.
- Filter rows with a boolean condition, and aggregate with `groupby` + `agg`.
- `merge` a lookup table onto a DataFrame and verify the row count survived the join.
- Assign engineered columns, and say why a whole-column expression beats `.apply()`.
- Save a feature table your next four labs can load.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- قراءة أنواع الأعمدة (`dtypes`) واكتشاف العمود الذي نوعه خاطئ قبل أن يُعطّل النموذج.
- تحويل عمود نصّي إلى رقمي، وشرح معنى القيم المفقودة الناتجة عن ذلك.
- ترشيح الصفوف بشرط منطقي، والتجميع باستخدام `groupby` مع `agg`.
- دمج جدول بحث بـ `merge` والتحقّق أن عدد الصفوف لم يتغيّر بعد الدمج.
- إضافة أعمدة مُهندَسة، وبيان لماذا يتفوّق التعبير على العمود كاملًا على `.apply()`.
- حفظ جدول خصائص تستطيع معاملك الأربعة القادمة تحميله.

</div>

## About the data

**Dataset:** `telco_churn` — IBM's telecom customer sample · CC0 (public domain) · 7,043 rows × 21 columns

Each row is **one customer** of a telecom company, at one moment in time. The columns describe who
they are (`gender`, `SeniorCitizen`, `Partner`, `Dependents`), what they bought (`PhoneService`,
`InternetService`, `Contract`, and seven add-on services), and what they pay (`MonthlyCharges`,
`TotalCharges`, `tenure` in months). The target, `Churn`, says whether that customer left.

Predicting it is worth doing because keeping a customer costs a fraction of winning a new one. A model
that flags the fifty most likely leavers this month is a retention team's whole workload — and the
same shape covers subscription cancellation, loan default, and employee attrition.

About **26.5%** of customers churned, so a model that always predicts "No" is already right almost
three times in four. Remember that number when you see your first accuracy score.

**Watch out:** `TotalCharges` is stored as **text**, and eleven of its values are blank. It looks like
a number in `head()` and behaves like a string in arithmetic. Section 2 starts there.

**This week returns to this dataset four more times.** That is deliberate — you go deeper instead of
restarting on something new every day.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `telco_churn` — عيّنة عملاء اتصالات من IBM · ملكية عامة · ٧٬٠٤٣ صفًا × ٢١ عمودًا

كل صف يمثّل **عميلًا واحدًا** لشركة اتصالات في لحظة زمنية واحدة. وتصف الأعمدة من هو العميل، وما اشتراه
(الهاتف، والإنترنت، ونوع العقد، وسبع خدمات إضافية)، وما يدفعه (`MonthlyCharges` و`TotalCharges` ومدّة
الاشتراك `tenure` بالأشهر). أما الهدف `Churn` فيقول هل غادر هذا العميل الخدمة.

ويهمّ التنبّؤ به لأن الاحتفاظ بعميل يكلّف جزءًا صغيرًا من تكلفة كسب عميل جديد. والنموذج الذي يُشير إلى
الخمسين الأكثر احتمالًا للمغادرة هذا الشهر يمثّل عمل فريق الاحتفاظ كاملًا — والشكل نفسه ينطبق على إلغاء
الاشتراكات وتعثّر القروض وترك الموظفين للعمل.

وقد غادر نحو **٢٦٫٥٪** من العملاء، فالنموذج الذي يقول «لا» دائمًا يكون مُصيبًا في ثلاث حالات من أربع
تقريبًا. تذكّر هذا الرقم عند رؤية أول قيمة دقة تحصل عليها.

**انتبه:** العمود `TotalCharges` مخزّن **كنص**، وإحدى عشرة قيمة فيه فارغة. يبدو رقمًا في `head()`
ويتصرّف كنص في الحساب. ومن هنا يبدأ القسم الثاني.

**ويعود هذا الأسبوع إلى هذه البيانات أربع مرات أخرى**، وذلك مقصود: فأنت تتعمّق بدل أن تبدأ من جديد كل يوم.

</div>

## Setup

Run the cell below first. It installs anything missing, fixes the random seed, and finds the dataset —
whether you are on your own machine or on Google Colab.

If it fails with `ModuleNotFoundError: aiep`, check that your kernel is **Python (aiep)**
(*Kernel → Change kernel*), or re-run `uv pip install -e shared/`.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. تُثبّت ما ينقص، وتُثبّت البذرة العشوائية، وتجد ملف البيانات — سواء كنت على
جهازك أو على Google Colab.

إذا ظهر الخطأ `ModuleNotFoundError: aiep` فتحقّق أن النواة المختارة هي **Python (aiep)**
(من قائمة *Kernel ← Change kernel*)، أو أعد تنفيذ الأمر `uv pip install -e shared/`.

</div>

In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    # A clone that never ran `uv pip install -e shared/` still has the package on disk —
    # use it before reaching for the network. Colab (no clone) falls through to pip.
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_shape, report

ensure("scikit-learn", "pyarrow")
seed_everything(42)

import numpy as np
import pandas as pd

DATA = get_dataset("telco_churn")
print(describe_dataset("telco_churn"))
print("\n", versions())

## Section 1 — Warm-up: read the types before you read the numbers  (≈25 min)

Everything in this section already works. Run it, read it, and answer the question at the end of each
cell in your head before you scroll on.

Three methods tell you almost everything about a table you have never seen: `info()` for the types
and the missing counts, `dtypes` for the types alone, and `describe(include='all')` for the shape of
every column, numeric or not. Most data bugs announce themselves in one of those three outputs.

<div dir="rtl" align="right">

## القسم الأول — التهيئة: اقرأ الأنواع قبل أن تقرأ الأرقام (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا. شغّله واقرأه، وأجب في ذهنك عن السؤال في نهاية كل خلية قبل أن تتابع.

ثلاث دوال تخبرك بمعظم ما تحتاجه عن جدول لم تره من قبل: `info()` للأنواع وعدد القيم المفقودة، و`dtypes`
للأنواع وحدها، و`describe(include='all')` لصورة كل عمود رقميًا كان أو نصيًا. ومعظم عيوب البيانات تُعلن
عن نفسها في أحد هذه المخرجات الثلاثة.

</div>

In [ ]:
df = pd.read_parquet(DATA)

print(f"rows: {df.shape[0]:,}   columns: {df.shape[1]}")
print(f"missing values reported by pandas: {df.isna().sum().sum()}")
df.head()

`info()` puts the type and the non-null count of every column in one place. Read the `Dtype` column
top to bottom before you read anything else.

Two columns hold money. Only one of them is a number.

<div dir="rtl" align="right">

تجمع `info()` نوع كل عمود وعدد قيمه غير الفارغة في مكان واحد. اقرأ عمود `Dtype` من أعلاه إلى أسفله قبل
أي شيء آخر.

هناك عمودان يحملان مبالغ مالية، وواحد منهما فقط رقم.

</div>

In [ ]:
df.info()

### Task 1.1 — the column that lies

`MonthlyCharges` is `float64`. `TotalCharges` is `string`. They are both money.

The cell below shows what that costs you: the numeric summary silently skips `TotalCharges`, and
adding the column to itself concatenates text instead of adding numbers.

Nothing here raises an error. That is the point — a wrong dtype does not crash, it just quietly
produces nonsense.

<div dir="rtl" align="right">

### المهمة ١٫١ — العمود الذي يكذب

العمود `MonthlyCharges` من نوع `float64`، والعمود `TotalCharges` من نوع نصّي، وكلاهما مبلغ مالي.

تُظهر الخلية أدناه ما يكلّفك ذلك: فالملخّص الرقمي يتجاوز `TotalCharges` بصمت، وجمع العمود على نفسه
يُلحق النصوص بدل أن يجمع الأرقام.

ولا شيء هنا يُخرج خطأ، وهذا هو المقصود: فالنوع الخاطئ لا يُعطّل البرنامج، بل يُنتج نتائج بلا معنى بهدوء.

</div>

In [ ]:
print("numeric columns describe() is willing to summarise:")
print(list(df.describe().columns))
print("\nTotalCharges is missing from that list. Its dtype is:", df["TotalCharges"].dtype)

print("\nMonthlyCharges + MonthlyCharges  ->", (df["MonthlyCharges"] + df["MonthlyCharges"]).iloc[0])
print("TotalCharges  + TotalCharges     ->", (df["TotalCharges"] + df["TotalCharges"]).iloc[0])

`describe(include='all')` includes the text columns too. For those it reports `unique`, `top` and
`freq` instead of `mean` and `std`, which is how you find a category column with 6,000 distinct
values or a column that is the same value all the way down.

**Change one thing:** the dataset has 21 columns, so the output is wide. Try
`df.describe(include='all').T` instead and see which shape you can actually read.

<div dir="rtl" align="right">

تشمل `describe(include='all')` الأعمدة النصية أيضًا، وتُبلّغ عنها بعدد القيم المختلفة والقيمة الأكثر
تكرارًا وعدد مرات ظهورها بدل المتوسط والانحراف المعياري — وبهذا تكتشف عمودًا فئويًا فيه ستة آلاف قيمة
مختلفة، أو عمودًا قيمته واحدة من أعلاه إلى أسفله.

**غيّر شيئًا واحدًا:** الجدول فيه ٢١ عمودًا فالمخرج عريض. جرّب `df.describe(include='all').T` وانظر أي
الشكلين تستطيع قراءته فعلًا.

</div>

In [ ]:
df.describe(include="all").T

**Do not fix `TotalCharges` yet.** Notice it, write down what you think is wrong with it, and carry
on. Section 2 opens by fixing it properly — and the fix turns out to teach more than the problem did.

<div dir="rtl" align="right">

**لا تُصلح `TotalCharges` الآن.** لاحظه واكتب ما تظنّه خطأً فيه ثم تابع. فالقسم الثاني يبدأ بإصلاحه
إصلاحًا صحيحًا، وسيتبيّن أن الإصلاح يُعلّمك أكثر مما علّمتك المشكلة.

</div>

## Section 2 — Core: six operations, one feature table  (≈60 min)

Six tasks. Each one is a pandas operation you will use for the rest of your career, applied to a
question a telecom company would actually ask.

1. Fix the type of `TotalCharges` — and explain the eleven missing values it creates.
2. Boolean filtering: who churned on a month-to-month contract?
3. `groupby` + `agg`: what is the churn rate by contract type?
4. `merge`: attach a lookup table, and check the join did not multiply your rows.
5. Assign two engineered columns.
6. Vectorisation: the same computation with `.apply()` and with a whole-column expression, timed.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ستّ عمليات وجدول خصائص واحد (نحو ٦٠ دقيقة)

ستّ مهام، كل واحدة منها عملية في pandas ستستخدمها طوال حياتك المهنية، ومطبَّقة على سؤال تسأله شركة
اتصالات فعلًا.

١. أصلح نوع العمود `TotalCharges`، واشرح القيم المفقودة الإحدى عشرة التي ينتجها الإصلاح.
٢. الترشيح المنطقي: من غادر من أصحاب العقود الشهرية؟
٣. `groupby` مع `agg`: ما نسبة المغادرة حسب نوع العقد؟
٤. `merge`: ألحق جدول بحث، وتحقّق أن الدمج لم يُضاعف صفوفك.
٥. أضف عمودين مُهندَسين.
٦. العمل على العمود كاملًا: الحساب نفسه بـ `.apply()` ثم بتعبير على العمود كله، مع قياس الزمن.

</div>

### Task 2.1 — convert `TotalCharges`, then ask why it broke

`pd.to_numeric` converts text to numbers. On this column it will hit values it cannot convert, and
`errors="coerce"` turns each of those into `NaN` rather than raising.

That is the easy half. The half that matters: **eleven rows become `NaN`. Find out what those eleven
customers have in common.** Do not accept "the data is dirty" — that is not an answer, it is a shrug.

Print the eleven rows and look at them before you read the next markdown cell.

<div dir="rtl" align="right">

### المهمة ٢٫١ — حوّل `TotalCharges` ثم اسأل لماذا انكسر

تحوّل الدالة `pd.to_numeric` النص إلى أرقام. وستقابل في هذا العمود قيمًا لا تستطيع تحويلها، ويجعل
الخيار `errors="coerce"` كل واحدة منها `NaN` بدل إخراج خطأ.

هذا هو النصف السهل. والنصف المهم: **إحدى عشرة قيمة تصبح `NaN`، فاكتشف ما يشترك فيه هؤلاء العملاء
الأحد عشر.** ولا تقبل جواب «البيانات وسِخة»، فهذا ليس جوابًا بل تهرّبًا.

اطبع الصفوف الأحد عشر وانظر إليها قبل قراءة خلية الشرح التالية.

</div>

In [ ]:

# TODO: Convert TotalCharges to a numeric column, coercing anything unconvertible to NaN.
# مهمة: حوّل العمود TotalCharges إلى عمود رقمي، مع تحويل غير القابل للتحويل إلى NaN.

# TODO: How many values became NaN?
# مهمة: كم قيمة أصبحت NaN؟
n_missing = ...

print(f"TotalCharges is now {df['TotalCharges'].dtype}")
print(f"rows that became NaN: {n_missing}")
print()

# TODO: Show the rows whose TotalCharges is now NaN, with tenure and MonthlyCharges.
# مهمة: اعرض الصفوف التي أصبح فيها TotalCharges قيمة NaN، مع tenure وMonthlyCharges.
missing_rows

Every one of the eleven has **`tenure = 0`**.

They are not corrupt rows. They are customers who signed up and have **not been billed yet**. There
is no total to record, so the field was left blank. The missingness has a cause, and the cause tells
you what to do about it: the right value is not the column mean and not a guess — it is **zero**,
because zero is what they have been charged.

Check that claim rather than trusting this paragraph — the cell below compares the two counts.

This is the habit the whole of W2D3 is built on. A missing value is a question, not a defect.

<div dir="rtl" align="right">

كل واحد من الأحد عشر عميلًا مدّة اشتراكه **`tenure = 0`**.

فهذه ليست صفوفًا معطوبة، بل عملاء اشتركوا و**لم تُصدر لهم فاتورة بعد**، فلا يوجد إجمالي ليُسجَّل فتُرك
الحقل فارغًا. إذًا للقيمة المفقودة سبب، والسبب يخبرك بما تفعله: القيمة الصحيحة ليست متوسط العمود ولا
تخمينًا، بل **صفر**، لأن صفرًا هو ما دُفع فعلًا.

تحقّق من هذا الادعاء بدل الوثوق بهذه الفقرة، فالخلية أدناه تقارن العددين.

وعلى هذه العادة يُبنى يوم الأسبوع الثاني الثالث كاملًا: القيمة المفقودة سؤال لا عيب.

</div>

In [ ]:

# TODO: Count the rows with tenure == 0, and check they are exactly the NaN rows.
# مهمة: احسب الصفوف التي فيها tenure == 0، وتحقّق أنها هي نفسها صفوف NaN.

print(f"NaN TotalCharges: {n_missing}   |   tenure == 0: {n_zero_tenure}")
print(f"the same rows, row for row: {same_rows}")

# TODO: Fill the missing TotalCharges with what the cause implies these customers were charged.
# مهمة: املأ قيم TotalCharges المفقودة بما يقتضيه السبب من مبلغ دُفع لهؤلاء العملاء.

print(f"missing after the fix: {df['TotalCharges'].isna().sum()}")

### Task 2.2 — boolean filtering

A boolean condition on a column gives you a `Series` of `True`/`False`, one per row. Put that inside
`df[...]` and you get the rows where it was `True`. Combine conditions with `&` (and) and `|` (or) —
and wrap each condition in brackets, because `&` binds tighter than `==` and Python will otherwise
compare the wrong things.

The business question: **how many of our churned customers were on month-to-month contracts?** If
that share is much higher than the month-to-month share of the customer base, the contract type is
not a detail, it is the story.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الترشيح المنطقي

الشرط المنطقي على عمود يعطيك سلسلة من `True` و`False` بقيمة لكل صف. وإذا وضعتها داخل `df[...]` حصلت
على الصفوف التي كانت قيمتها `True`. وتُجمع الشروط بـ `&` بمعنى «و» وبـ `|` بمعنى «أو» — مع وضع كل شرط
بين قوسين، لأن `&` أقوى ارتباطًا من `==` وإلا قارنت بايثون أشياء غير التي تريد.

والسؤال العملي: **كم من عملائنا المغادرين كان على عقد شهري؟** فإذا كانت هذه النسبة أعلى بكثير من نسبة
أصحاب العقود الشهرية بين العملاء كلهم، فنوع العقد ليس تفصيلًا بل هو القصة نفسها.

</div>

In [ ]:

# TODO: Select the churned customers who were on a month-to-month contract.
# مهمة: اختر العملاء المغادرين الذين كانوا على عقد شهري.
churned_m2m = ...

all_churned = df[df["Churn"] == "Yes"]
share_of_churners = len(churned_m2m) / len(all_churned)
share_of_base = (df["Contract"] == "Month-to-month").mean()

print(f"churned on month-to-month: {len(churned_m2m):,} of {len(all_churned):,} churners")
print(f"  that is {share_of_churners:.1%} of everyone who left")
print(f"  month-to-month is only {share_of_base:.1%} of the customer base")

### Task 2.3 — `groupby` + `agg`

Three lines, and they produce the single most predictive fact in this dataset.

`groupby("Contract")` splits the table into one group per contract type. `agg` then computes whatever
you ask for **per group**. Passing a dictionary — `{"column": ["mean", "count"]}` — lets you ask for
several statistics at once, which is almost always what you want: a rate without its group size is
not interpretable.

To average a Yes/No column, turn it into 1/0 first. The mean of a 0/1 column **is** the rate.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — `groupby` مع `agg`

ثلاثة أسطر تُنتج أقوى حقيقة تنبؤية في هذه البيانات.

تقسم `groupby("Contract")` الجدول إلى مجموعة لكل نوع عقد، ثم تحسب `agg` ما تطلبه **لكل مجموعة**.
وتمرير قاموس مثل `{"column": ["mean", "count"]}` يتيح طلب عدة مقاييس مرة واحدة، وهذا هو المطلوب عادةً:
فالنسبة بلا حجم مجموعتها غير قابلة للتفسير.

ولحساب متوسط عمود قيمه «نعم/لا» حوّله أولًا إلى ١ و٠، فمتوسط عمود من الأصفار والوحدات **هو** النسبة نفسها.

</div>

In [ ]:

# TODO: Add a churn_flag column: 1 where Churn is "Yes", 0 otherwise.
# مهمة: أضف عمودًا churn_flag قيمته ١ حيث Churn تساوي "Yes" و٠ فيما عدا ذلك.

# TODO: Churn rate and customer count per contract type, worst rate first.
# مهمة: نسبة المغادرة وعدد العملاء لكل نوع عقد، وأسوأ نسبة أولًا.

print(churn_by_contract.to_string(float_format=lambda v: f"{v:,.3f}"))
print(f"\noverall churn rate: {df['churn_flag'].mean():.3f}")

Month-to-month customers churn at **42.7%**. Two-year customers churn at **2.8%** — fifteen times
lower. Nothing else in this dataset separates the two groups that cleanly.

Be careful about what that licenses you to say. This is not evidence that moving someone onto a
two-year contract stops them leaving; people who expect to stay are the ones who sign long contracts
in the first place. The number is enormously useful for *prediction* and says almost nothing about
*cause*. W2D2 comes back to exactly this distinction.

<div dir="rtl" align="right">

يغادر أصحاب العقود الشهرية بنسبة **٤٢٫٧٪**، وأصحاب عقود السنتين بنسبة **٢٫٨٪**، أي أقل بخمسة عشر
ضعفًا. ولا يوجد في هذه البيانات ما يفصل المجموعتين بهذا الوضوح.

لكن انتبه لما يسمح لك هذا الرقم بقوله. فهو ليس دليلًا على أن نقل العميل إلى عقد سنتين يمنعه من
المغادرة؛ لأن من يتوقّع البقاء هو من يوقّع عقدًا طويلًا من الأصل. الرقم مفيد جدًا في **التنبّؤ** ولا
يقول شيئًا تقريبًا عن **السبب**. ويعود الأسبوع الثاني اليوم الثاني إلى هذا التمييز بالتحديد.

</div>

### Task 2.4 — `merge`, and the row count that proves it worked

A `merge` attaches columns from a second table by matching on a shared key. It is the same operation
as a SQL `JOIN`, and it has the same failure mode: **if the key is not unique in the right-hand table,
your rows multiply.** A 7,043-row table joined against a lookup with a duplicated key comes back with
9,000 rows, no error, and every downstream number quietly wrong.

So the rule is mechanical: check `len(df)` before, check it after, and make them match.

Here you build a three-row lookup giving each contract type its length in months, and join it on.
`how="left"` keeps every left-hand row whether or not it found a match — which is what you want, and
which is also what leaves `NaN` behind when a key is missing from the lookup.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الدمج، وعدد الصفوف الذي يُثبت نجاحه

يُلحق `merge` أعمدةً من جدول ثانٍ بمطابقة مفتاح مشترك. وهو العملية نفسها التي تسمّى `JOIN` في SQL، وله
نمط الفشل نفسه: **إذا لم يكن المفتاح فريدًا في الجدول الأيمن تضاعفت صفوفك.** فجدول من ٧٬٠٤٣ صفًا يُدمج
بجدول بحث فيه مفتاح مكرّر يعود بتسعة آلاف صف، بلا أي خطأ، وبكل رقم لاحق خاطئ بهدوء.

فالقاعدة آليّة: تحقّق من `len(df)` قبل الدمج وبعده، واجعلهما متساويين.

وهنا تبني جدول بحث من ثلاثة صفوف يعطي كل نوع عقد طوله بالأشهر، ثم تدمجه. ويُبقي `how="left"` كل صفوف
الجدول الأيسر سواء وجدت مطابقًا أم لا — وهذا هو المطلوب، وهو أيضًا ما يترك قيم `NaN` عندما يغيب مفتاح
عن جدول البحث.

</div>

In [ ]:

rows_before = len(df)

# TODO: Build the three-row lookup table mapping each contract type to its length in months.
# مهمة: ابنِ جدول البحث من ثلاثة صفوف الذي يربط كل نوع عقد بطوله بالأشهر.

# TODO: Merge it onto df on Contract, keeping every row of df.
# مهمة: ادمجه على df عبر عمود Contract مع الإبقاء على كل صفوف df.
df = ...

rows_after = len(df)
print(f"rows before: {rows_before:,}   after: {rows_after:,}")
print(f"unmatched rows (NaN in the new column): {df['contract_months'].isna().sum()}")

### Task 2.5 — two engineered columns

A feature is a hypothesis written as arithmetic. Two of them here:

- **`avg_monthly_spend`** — `TotalCharges / tenure`. The hypothesis: what a customer pays *on average
  over their whole life* differs from what they pay *this month*, and the gap says something about
  price rises and added services. Careful: eleven customers have `tenure = 0`, and dividing by zero
  gives `inf`, which will poison every model you fit. Those eleven pay exactly one month's charge, so
  `MonthlyCharges` is the honest value for them.
- **`tenure_bucket`** — `tenure` cut into `0-12`, `13-24`, `25-48`, `49+` months. The hypothesis: the
  risk of leaving is not a straight line in tenure; the first year is its own thing. Buckets let a
  linear model express that.

Then check both: no `NaN`, no `inf`.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — عمودان مُهندَسان

الخاصية فرضية مكتوبة على شكل حساب، وهنا فرضيتان:

- **`avg_monthly_spend`** أي `TotalCharges / tenure`. والفرضية: أن ما يدفعه العميل **في المتوسط طوال
  مدّته** يختلف عمّا يدفعه **هذا الشهر**، وأن الفرق يقول شيئًا عن ارتفاع الأسعار والخدمات المضافة.
  وانتبه: أحد عشر عميلًا مدّتهم صفر، والقسمة على صفر تعطي `inf` وهي تُفسد أي نموذج تُدرّبه. وهؤلاء
  الأحد عشر يدفعون قيمة شهر واحد بالضبط، فقيمة `MonthlyCharges` هي القيمة الصادقة لهم.
- **`tenure_bucket`** أي تقسيم `tenure` إلى فئات `0-12` و`13-24` و`25-48` و`49+` شهرًا. والفرضية: أن
  خطر المغادرة ليس خطًا مستقيمًا مع المدّة، وأن السنة الأولى حالة خاصة. والفئات تتيح للنموذج الخطّي
  التعبير عن ذلك.

ثم تحقّق من العمودين: لا `NaN` ولا `inf`.

</div>

In [ ]:

# TODO: Average spend per month of tenure, falling back to MonthlyCharges when tenure is 0.
# مهمة: متوسط الإنفاق لكل شهر من المدّة، مع الرجوع إلى MonthlyCharges عندما تكون المدّة صفرًا.

# TODO: Cut tenure into the four labelled bands.
# مهمة: قسّم المدّة إلى النطاقات الأربعة المُسمّاة.

engineered = ["avg_monthly_spend", "tenure_bucket"]
print(df[engineered + ["tenure", "TotalCharges", "MonthlyCharges"]].head(3).to_string())
print(f"\nNaN in the engineered columns: {df[engineered].isna().sum().sum()}")
print(f"infinite values: {np.isinf(df['avg_monthly_spend']).sum()}")
print(f"\n{df['tenure_bucket'].value_counts().to_string()}")

### Task 2.6 — the same computation, two ways, timed

`.apply()` looks like pandas and behaves like a Python `for` loop: it calls your function once per
row, in the interpreter. A whole-column expression hands the same work to compiled code that operates
on the entire array at once. That is **vectorisation**, and on this small table it is worth a factor of
a few hundred — measure it rather than taking that number on trust.

Compute the same thing both ways — a charge-to-tenure ratio — confirm the answers are identical, and
time both with `%timeit`. Report the ratio.

The number matters less than the reflex: when you find yourself writing `.apply()`, ask what the
whole-column version looks like first.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الحساب نفسه بطريقتين، مع قياس الزمن

تبدو `.apply()` من pandas وتتصرّف كحلقة `for` في بايثون: فهي تنادي دالتك مرة لكل صف داخل المُفسِّر. أما
التعبير على العمود كاملًا فيُسلّم العمل نفسه إلى شيفرة مُصرَّفة تعمل على المصفوفة كلها مرة واحدة. وهذا
هو **العمل على المتّجهات**، ويساوي في هذا الجدول الصغير بضع مئات من الأضعاف — فقِسه بنفسك ولا تأخذ
هذا الرقم على الثقة.

احسب الشيء نفسه بالطريقتين — نسبة المبلغ إلى المدّة — وتأكّد أن الناتجين متطابقان، وقِس زمنيهما بـ
`%timeit`، ثم اذكر النسبة بينهما.

والرقم أقل أهمية من العادة: فحين تجد نفسك تكتب `.apply()`، اسأل أولًا كيف تكون صيغة العمود الكامل.

</div>

In [ ]:

# TODO: The row-by-row version, with .apply() along axis=1.
# مهمة: الصيغة صفًا صفًا باستخدام `.apply()` على المحور axis=1.
by_apply = ...

# TODO: The whole-column version — no apply, no loop.
# مهمة: صيغة العمود الكامل، بلا apply وبلا حلقة.
by_vector = ...

print("identical results:", np.allclose(by_apply, by_vector))

t_apply = %timeit -o -q -n 3 -r 3 df.apply(lambda row: row["MonthlyCharges"] * row["tenure"], axis=1)
t_vector = %timeit -o -q -n 3 -r 3 df["MonthlyCharges"] * df["tenure"]

# TODO: Print both timings and how many times faster the vectorised version is.
# مهمة: اطبع الزمنين وكم مرة تكون صيغة العمود الكامل أسرع.

## Section 3 — Stretch: which single column separates churners best?  (≈30 min)

Open-ended. Lower expectation of completeness — get through the first part, and treat the second as
homework if you run out of time.

You have seen that `Contract` splits churn 42.7% against 2.8%. Now check every other column and find
out whether anything beats it.

For a **numeric** column, compare its mean among churners against its mean among stayers, and scale
the gap by the column's own spread — otherwise `TotalCharges`, which is measured in thousands, wins
every comparison by units alone. For a **categorical** column, the churn rate of its best and worst
category is the same idea.

Write your answer down. **D2 will confirm it or contradict it visually**, and it is worth knowing
which of those happened before you see the charts.

Then the harder question: a column that separates the classes well is not the same as a column a
retention team can act on. Which of your top three could someone actually do something about?

**Link to your capstone:** this is the first version of a question your capstone report has to answer
about its own data — not "what does my model score" but "what in the data is carrying the score".

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: أي عمود واحد يفصل المغادرين أفضل من غيره؟ (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل — أنجز الجزء الأول واعتبر الثاني واجبًا منزليًا إن ضاق الوقت.

رأيت أن العمود `Contract` يفصل المغادرة بنسبة ٤٢٫٧٪ مقابل ٢٫٨٪. فافحص الآن كل عمود آخر واكتشف هل يتفوّق
عليه شيء.

وفي العمود **الرقمي** قارن متوسطه بين المغادرين ومتوسطه بين الباقين، واقسم الفرق على انحراف العمود
نفسه، وإلا فاز `TotalCharges` المقيس بالآلاف في كل مقارنة بحكم وحدته وحدها. وفي العمود **الفئوي** تكون
نسبة المغادرة في أفضل فئة وأسوأ فئة هي الفكرة نفسها.

اكتب إجابتك. **وسيؤكّدها أو يخالفها اليوم الثاني بصريًا**، ومن المفيد أن تعرف أيّ الأمرين حدث قبل أن
ترى الرسوم.

ثم السؤال الأصعب: العمود الذي يفصل الفئات جيدًا ليس بالضرورة عمودًا يستطيع فريق الاحتفاظ التصرّف
بناءً عليه. فأيّ الأعمدة الثلاثة الأولى لديك يمكن لأحد أن يفعل به شيئًا فعلًا؟

**الصلة بمشروعك:** هذه أول صورة لسؤال يجب أن يجيب عنه تقرير مشروعك عن بياناته الخاصة — لا «كم نتيجة
نموذجي» بل «ما الذي في البيانات يحمل هذه النتيجة».

</div>

In [ ]:

# TODO: Rank the numeric columns by how far apart the two groups' means are, in standard deviations.
# مهمة: رتّب الأعمدة الرقمية بمقدار تباعد متوسطي المجموعتين مقيسًا بالانحرافات المعيارية.

# TODO: Now the categorical columns: the churn-rate spread within each one.
# مهمة: والآن الأعمدة الفئوية: مدى تفاوت نسبة المغادرة داخل كل عمود.

**Your answer:** _(which column separates churners best, and could a retention team act on it? Write
two or three sentences. Keep them — D2 asks you to compare them against what the charts show.)_

<div dir="rtl" align="right">

**إجابتك:** _(أي عمود يفصل المغادرين أفضل من غيره، وهل يستطيع فريق الاحتفاظ التصرّف بناءً عليه؟ اكتب
جملتين أو ثلاثًا، واحتفظ بهما — فاليوم الثاني يطلب منك مقارنتهما بما تُظهره الرسوم.)_

</div>

## Save your artefact

`features.parquet` is the feature table the rest of the week loads: D2 charts it, D5 builds a
`Pipeline` on it. Parquet rather than CSV, for one reason that matters here — it stores the dtype of
every column, so the `TotalCharges` you spent Section 2 fixing comes back as a number tomorrow
instead of as text again.

Everything a lab produces goes into `artefacts/`, which is not committed to git. If you ever lose it,
re-run the notebook.

<div dir="rtl" align="right">

## احفظ مخرجاتك

الملف `features.parquet` هو جدول الخصائص الذي تُحمّله بقية أيام الأسبوع: يرسمه اليوم الثاني، ويبني عليه
اليوم الخامس خطّ معالجة (`Pipeline`). وهو بصيغة parquet لا csv لسبب مهم هنا: أنه يحفظ نوع كل عمود، فيعود
العمود `TotalCharges` الذي أصلحته في القسم الثاني رقمًا غدًا لا نصًا من جديد.

وكل ما ينتجه المعمل يذهب إلى مجلّد `artefacts/` غير المُدرج في git. وإذا فقدته فأعد تشغيل الدفتر.

</div>

In [ ]:
out = ARTEFACT_DIR / "features.parquet"
df.to_parquet(out, index=False)

# Read it back before trusting it. A file that does not round-trip is not an artefact.
reloaded = pd.read_parquet(out)
print(f"Saved {out}")
print(f"{reloaded.shape[0]:,} rows x {reloaded.shape[1]} columns")
print(f"round-trips to an equal DataFrame: {reloaded.equals(df)}")
print(f"TotalCharges comes back as: {reloaded['TotalCharges'].dtype}")

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

</div>

In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check(pd.api.types.is_numeric_dtype(df["TotalCharges"]),
      f"TotalCharges must be numeric, it is {df['TotalCharges'].dtype}",
      f"يجب أن يكون العمود TotalCharges رقميًا، ونوعه الحالي {df['TotalCharges'].dtype}")

check(n_missing == n_zero_tenure == 11,
      f"the coercion should produce exactly 11 NaNs, matching the 11 tenure-0 customers "
      f"(got {n_missing} NaNs and {n_zero_tenure} zero-tenure rows)",
      f"يجب أن يُنتج التحويل إحدى عشرة قيمة NaN بعدد عملاء المدّة صفر "
      f"(الناتج {n_missing} قيمة و{n_zero_tenure} صفًا)")

check(churn_by_contract.shape[0] == df["Contract"].nunique(),
      f"the groupby should give one row per contract type "
      f"({df['Contract'].nunique()} expected, got {churn_by_contract.shape[0]})",
      f"يجب أن يعطي التجميع صفًا واحدًا لكل نوع عقد "
      f"(المتوقّع {df['Contract'].nunique()} والناتج {churn_by_contract.shape[0]})")

check(rows_after == rows_before,
      f"the merge must not change the row count ({rows_before:,} before, {rows_after:,} after) — "
      f"if it grew, the lookup has a duplicated key",
      f"يجب ألا يغيّر الدمج عدد الصفوف ({rows_before:,} قبل و{rows_after:,} بعد) — "
      f"وإن زاد فجدول البحث فيه مفتاح مكرّر")

check(df[["avg_monthly_spend", "tenure_bucket"]].isna().sum().sum() == 0
      and not np.isinf(df["avg_monthly_spend"]).any(),
      f"the engineered columns must have no NaN and no inf "
      f"(NaN: {df[['avg_monthly_spend', 'tenure_bucket']].isna().sum().sum()}, "
      f"inf: {int(np.isinf(df['avg_monthly_spend']).sum())})",
      f"يجب ألا تحتوي الأعمدة المُهندَسة NaN ولا inf "
      f"(NaN: {df[['avg_monthly_spend', 'tenure_bucket']].isna().sum().sum()}، "
      f"inf: {int(np.isinf(df['avg_monthly_spend']).sum())})")

check(np.allclose(by_apply, by_vector),
      "the .apply() and vectorised versions must give the same answer",
      "يجب أن تعطي صيغة `.apply()` وصيغة العمود الكامل الجواب نفسه")

check(reloaded.equals(df),
      "features.parquet must round-trip to a DataFrame equal to the one you saved",
      "يجب أن يعود الملف features.parquet إلى جدول مساوٍ للجدول الذي حفظته")

report()

## What's next

Tomorrow (**W2D2**) you load this `features.parquet` and look at it — five charts, and for each one a
written sentence saying which decision it changed. Your answer from Section 3 goes on trial: the
column you picked either shows up clearly in a chart, or it does not, and both outcomes teach you
something about ranking features with group means.

<div dir="rtl" align="right">

## ماذا بعد

غدًا (**الأسبوع ٢ اليوم ٢**) تُحمّل هذا الملف `features.parquet` وتنظر إليه: خمسة رسوم، ولكل رسم جملة
مكتوبة تقول أي قرار غيّره. وستُختبر إجابتك من القسم الثالث: فالعمود الذي اخترته إمّا يظهر بوضوح في رسم
أو لا يظهر، وكلا الاحتمالين يُعلّمك شيئًا عن ترتيب الخصائص بمتوسطات المجموعات.

</div>